# Role-Aware SAAMR: All-Atom PE/PEAA Ionomer to OpenMM

**Author:** Joseph R. Laforet Jr.

This notebook builds all-atom analogues of the PE/PEAA ionomer systems described by the coarse-grained paper schematic. The paper simulated a CG representation; here MuPT builds chemically explicit polyethylene/acrylic-acid-side-chain polymers with explicit sodium counterions and sends the resulting system toward OpenFF/OpenMM.

The notebook is intentionally knob-driven. It defaults to a small smoke test, while the production target is 800 chains per system.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import sys
import json

import networkx as nx
import numpy as np
from rdkit import Chem
try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else range(0)


def find_examples_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in (start, *start.parents):
        if (candidate / "examples_system").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate the mupt-examples repository root")


EXAMPLES_ROOT = find_examples_root()
LOCAL_MUPT_SOURCE = EXAMPLES_ROOT / "mupt"
if LOCAL_MUPT_SOURCE.exists():
    sys.path.insert(0, str(LOCAL_MUPT_SOURCE))

from mupt.builders.random_walk import AngleConstrainedRandomWalk
from mupt.geometry.coordinates.reference import origin
from mupt.geometry.shapes import Ellipsoid, PointCloud
from mupt.geometry.transforms.rigid import rigid_vector_coalignment
from mupt.interfaces.rdkit import primitive_from_mupt_sdf, primitive_to_rdkit_mols
from mupt.interfaces.smiles import primitive_from_smiles
from mupt.mupr.primitives import Primitive
from mupt.mupr.topology import TopologicalStructure
from mupt.roles import PrimitiveRole

OUTPUT_ROOT = EXAMPLES_ROOT / "examples_system" / "role_aware_ionomer_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {EXAMPLES_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")

Repository root: /home/joelaforet/Shirts-Lab-Linux/mupt-examples
Output root: /home/joelaforet/Shirts-Lab-Linux/mupt-examples/examples_system/role_aware_ionomer_outputs


## 1. Science Knobs

`USE_PRODUCTION_SIZE = False` keeps the notebook runnable as a demonstration. Set it to `True` to build 800 chains for the selected system.

In [2]:
SYSTEM_SPECS = {
    "m3": {
        "pattern": ["PE", "PEAA", "PE"],
        "repeat_count": 12,
        "expected_units": 36,
    },
    "m5": {
        "pattern": ["PE", "PE", "PEAA", "PE", "PE"],
        "repeat_count": 7,
        "expected_units": 35,
    },
    "m7": {
        "pattern": ["PE", "PE", "PE", "PEAA", "PE", "PE", "PE"],
        "repeat_count": 5,
        "expected_units": 35,
    },
}

BUILD_SYSTEM_NAME = "m3"  # "m3", "m5", "m7", or "all"
USE_PRODUCTION_SIZE = False
N_CHAINS_SMOKE_TEST = 2
N_CHAINS_PRODUCTION = 100
RANDOM_SEED = 51

INTER_RESIDUE_BOND_LENGTH_A = 2.8
INITIAL_CHAIN_GRID_SPACING_A = 18.0
INITIAL_CHAIN_GRID_JITTER_FRACTION = 0.15
ANGLE_MAX_RAD = np.pi / 4
SODIUM_COO_DISTANCE_A = 2.4
MIN_ALLOWED_DISTANCE_A = 0.75

N_CHAINS = N_CHAINS_PRODUCTION if USE_PRODUCTION_SIZE else N_CHAINS_SMOKE_TEST
SELECTED_SYSTEMS = list(SYSTEM_SPECS) if BUILD_SYSTEM_NAME == "all" else [BUILD_SYSTEM_NAME]

print(f"Selected systems: {SELECTED_SYSTEMS}")
print(f"Chains per system: {N_CHAINS}")

Selected systems: ['m3']
Chains per system: 100


## 2. Repeat Chemistry

Each `PE` or `PEAA` repeat represents a three-carbon backbone block. The acid-bearing repeat is `-CH2-CH(CH2COO-)-CH2-`. Head and tail cap residues have one linker site and explicit terminal hydrogen caps; they do not add extra carbons. Sodium is represented as a separate one-particle residue and is not covalently bonded to the polymer.

In [3]:
REPEAT_SMILES = {
    "PE_HEAD": "[H]-[CH2:1]-[CH2]-[CH2:2]-*",
    "PE": "*-[CH2:1]-[CH2]-[CH2:2]-*",
    "PEAA": "*-[CH2:1]-[CH]([CH2][C](=O)[O-])-[CH2:2]-*",
    "PE_TAIL": "*-[CH2:1]-[CH2]-[CH2:2]-[H]",
    "NA": "[Na+]",
}

# Capped analogues are used only to obtain local conformers for the linker-bearing
# repeat templates. The MuPT topology is still built from REPEAT_SMILES above.
REPEAT_CONFORMER_SMILES = {
    "PE_HEAD": "[H]-[CH2:1]-[CH2]-[CH2:2]-[H]",
    "PE": "[H]-[CH2:1]-[CH2]-[CH2:2]-[H]",
    "PEAA": "[H]-[CH2:1]-[CH]([CH2][C](=O)[O-])-[CH2:2]-[H]",
    "PE_TAIL": "[H]-[CH2:1]-[CH2]-[CH2:2]-[H]",
}

RESNAME_MAP = {
    "PE_HEAD": "PEH",
    "PE": "PEX",
    "PEAA": "PAA",
    "PE_TAIL": "PET",
    "NA": "SOD",
}

PEAA_CARBOXYLATE_CARBON_ATOM_LABEL = 4
PEAA_CARBOXYLATE_OXYGEN_ATOM_LABELS = (5, 6)
AXIS = 0
SEMIMINOR_FRACT = 0.5


## 3. Notebook-Local Builder

This is intentionally notebook-local. The point is to demonstrate that MuPT can already represent the role-aware hierarchy, cap residues, counterions, SDF export, and OpenMM handoff without waiting for a specialized public ionomer builder API.

In [4]:
@dataclass(frozen=True)
class BuiltIonomerSystem:
    primitive: Primitive
    system_name: str
    chain_sequences: list[list[str]]
    sodium_count: int
    sdf_paths: list[Path]


def template_positions_from_smiles(smiles: str, random_seed: int = 52) -> dict[int, np.ndarray]:
    """Return RDKit conformer positions keyed by RDKit atom index."""
    template = primitive_from_smiles(
        smiles,
        label="template",
        ensure_explicit_Hs=True,
        embed_positions=True,
    )
    return {
        int(atom.label): np.array(atom.shape.centroid, dtype=float)
        for atom in template.children
    }


def assign_repeat_template_geometry(residue: Primitive, conformer_smiles: str) -> None:
    """Assign RDKit-generated local coordinates and an envelope shape to a repeat."""
    positions_by_label = template_positions_from_smiles(conformer_smiles)
    points = []
    for atom in residue.children:
        position = positions_by_label[int(atom.label)]
        atom.shape = PointCloud(position)
        points.append(position)
    residue.shape = PointCloud(np.vstack(points))

    head_atom, tail_atom = residue.search_hierarchy_by(
        lambda prim: "molAtomMapNumber" in prim.metadata,
        min_count=2,
    )
    head_pos = np.array(head_atom.shape.centroid, dtype=float)
    tail_pos = np.array(tail_atom.shape.centroid, dtype=float)
    major_radius = np.linalg.norm(tail_pos - head_pos) / 2.0
    axis_vec = np.zeros(3, dtype=float)
    axis_vec[AXIS] = major_radius
    residue.rigidly_transform(
        rigid_vector_coalignment(
            vector1_start=head_pos,
            vector1_end=tail_pos,
            vector2_start=origin(3),
            vector2_end=axis_vec,
            t1=0.5,
            t2=0.0,
        )
    )

    semiminor = SEMIMINOR_FRACT * major_radius
    radii = np.full(3, semiminor)
    radii[AXIS] = major_radius
    residue.shape = Ellipsoid(radii)

    for conn_handle, conn_ref in residue.external_connectors.items():
        atom = residue.children_by_handle[conn_ref.primitive_handle]
        anchor = np.array(atom.shape.centroid, dtype=float)
        direction = -1.0 if atom.metadata.get("molAtomMapNumber") == 1 else 1.0
        linker = anchor + np.array([direction * INTER_RESIDUE_BOND_LENGTH_A, 0.0, 0.0])
        for conn in (residue.fetch_connector(conn_handle), residue.fetch_connector_on_child(conn_ref)):
            conn.anchor.position = anchor
            conn.linker.position = linker


def residue_atom_position(residue: Primitive, atom_label: int) -> np.ndarray:
    """Return the centroid of the atom with the requested RDKit atom label."""
    for atom in residue.children:
        if int(atom.label) == atom_label:
            return np.array(atom.shape.centroid, dtype=float)
    raise KeyError(f"Could not find atom label {atom_label} in {residue.label}")


def rng_unit_vector(rng: np.random.Generator) -> np.ndarray:
    """Return a deterministic random unit vector from this notebook's RNG."""
    vector = rng.normal(size=3)
    return vector / np.linalg.norm(vector)


def compact_chain_start_point(chain_idx: int, n_chains: int, rng: np.random.Generator) -> np.ndarray:
    """Return a compact lattice start point instead of an expanding radial shell."""
    grid_dim = int(np.ceil(n_chains ** (1.0 / 3.0)))
    ix = chain_idx % grid_dim
    iy = (chain_idx // grid_dim) % grid_dim
    iz = chain_idx // (grid_dim * grid_dim)
    grid_center = 0.5 * (grid_dim - 1)
    point = INITIAL_CHAIN_GRID_SPACING_A * np.array([ix - grid_center, iy - grid_center, iz - grid_center], dtype=float)
    jitter = INITIAL_CHAIN_GRID_JITTER_FRACTION * INITIAL_CHAIN_GRID_SPACING_A * rng.normal(size=3)
    return point + jitter


def set_single_atom_residue_position(residue: Primitive, position: np.ndarray) -> None:
    """Move a one-particle residue to an absolute position."""
    atom = residue.children[0]
    atom.shape = PointCloud(np.array(position, dtype=float))
    residue.shape = PointCloud(np.array(position, dtype=float))


def place_sodium_near_peaa(peaa_residue: Primitive, rng: np.random.Generator) -> np.ndarray:
    """Return a sodium position near the carboxylate oxygen midpoint."""
    oxy_1 = residue_atom_position(peaa_residue, PEAA_CARBOXYLATE_OXYGEN_ATOM_LABELS[0])
    oxy_2 = residue_atom_position(peaa_residue, PEAA_CARBOXYLATE_OXYGEN_ATOM_LABELS[1])
    carbon = residue_atom_position(peaa_residue, PEAA_CARBOXYLATE_CARBON_ATOM_LABEL)
    midpoint = 0.5 * (oxy_1 + oxy_2)
    direction = midpoint - carbon
    if np.linalg.norm(direction) < 1.0e-8:
        direction = rng.normal(size=3)
    direction = direction / np.linalg.norm(direction)
    jitter = 0.05 * rng.normal(size=3)
    direction = direction + jitter
    direction = direction / np.linalg.norm(direction)
    return midpoint + SODIUM_COO_DISTANCE_A * direction


def build_repeat_lexicon() -> dict[str, Primitive]:
    """Create RESIDUE -> PARTICLE primitives for repeat units and sodium."""
    lexicon = {}
    for name, smiles in REPEAT_SMILES.items():
        residue = primitive_from_smiles(
            smiles,
            label=name,
            ensure_explicit_Hs=True,
            embed_positions=False,
        )
        residue.role = PrimitiveRole.RESIDUE
        residue.metadata.update({
            "repeat_kind": name,
            "residue_name": RESNAME_MAP[name],
        })
        for atom in residue.children:
            atom.role = PrimitiveRole.PARTICLE
            atom.metadata.setdefault("residue_name", RESNAME_MAP[name])
        if name != "NA":
            assign_repeat_template_geometry(residue, REPEAT_CONFORMER_SMILES[name])
        lexicon[name] = residue
    return lexicon


def expanded_sequence(system_name: str) -> list[str]:
    """Expand a system pattern and replace terminal PE blocks with cap residues."""
    spec = SYSTEM_SPECS[system_name]
    sequence = list(spec["pattern"]) * int(spec["repeat_count"])
    if len(sequence) != int(spec["expected_units"]):
        raise ValueError(f"{system_name} produced {len(sequence)} units, expected {spec['expected_units']}")
    if sequence[0] != "PE" or sequence[-1] != "PE":
        raise ValueError("Current cap logic expects chain patterns to start and end with PE")
    sequence[0] = "PE_HEAD"
    sequence[-1] = "PE_TAIL"
    return sequence


def build_ionomer_system(system_name: str, n_chains: int, random_seed: int) -> BuiltIonomerSystem:
    """Build one all-atom PE/PEAA ionomer system with explicit sodium counterions."""
    rng = np.random.default_rng(random_seed)
    np.random.seed(random_seed)
    lexicon = build_repeat_lexicon()
    sequence_template = expanded_sequence(system_name)

    universe = Primitive(label=f"ionomer_{system_name}", role=PrimitiveRole.UNIVERSE)
    universe.metadata.update({
        "system_name": system_name,
        "n_chains": str(n_chains),
        "terminal_caps": "explicit_hydrogen_residues",
    })

    chain_sequences = []
    sodium_segments = []
    sodium_count = 0

    for chain_idx in tqdm(range(n_chains), desc=f"building {system_name} chains", unit="chain"):
        segment = Primitive(label=f"chain_{chain_idx:04d}", role=PrimitiveRole.SEGMENT)
        segment.metadata.update({"system_name": system_name, "chain_index": str(chain_idx)})
        residue_handles = []
        chain_sequences.append(sequence_template)

        for repeat_idx, repeat_kind in enumerate(sequence_template):
            residue = lexicon[repeat_kind].copy()
            residue.label = f"repeat_{repeat_idx:03d}_{repeat_kind}"
            residue.role = PrimitiveRole.RESIDUE
            residue.metadata.update({
                "system_name": system_name,
                "chain_index": str(chain_idx),
                "repeat_index": str(repeat_idx),
                "repeat_kind": repeat_kind,
                "residue_name": RESNAME_MAP[repeat_kind],
            })
            for atom in residue.children:
                atom.role = PrimitiveRole.PARTICLE
                atom.metadata.update({
                    "system_name": system_name,
                    "chain_index": str(chain_idx),
                    "repeat_index": str(repeat_idx),
                    "repeat_kind": repeat_kind,
                    "residue_name": RESNAME_MAP[repeat_kind],
                })
            residue_handles.append(segment.attach_child(residue))

        segment.set_topology(
            nx.path_graph(residue_handles, create_using=TopologicalStructure),
            max_registration_iter=100,
        )
        direction = rng_unit_vector(rng)
        placement = AngleConstrainedRandomWalk(
            bond_length=INTER_RESIDUE_BOND_LENGTH_A,
            angle_max_rad=ANGLE_MAX_RAD,
            initial_point=compact_chain_start_point(chain_idx, n_chains, rng),
            initial_direction=direction,
        )
        for handle, transform in placement.generate_placements(segment):
            segment.children_by_handle[handle].rigidly_transform(transform)
        universe.attach_child(segment)

        for repeat_idx, repeat_kind in enumerate(sequence_template):
            if repeat_kind != "PEAA":
                continue
            peaa_residue = segment.children[repeat_idx]
            sodium = lexicon["NA"].copy()
            sodium.label = f"sodium_chain{chain_idx:04d}_repeat{repeat_idx:03d}"
            sodium.role = PrimitiveRole.RESIDUE
            sodium_position = place_sodium_near_peaa(peaa_residue, rng)
            set_single_atom_residue_position(sodium, sodium_position)
            sodium.metadata.update({
                "system_name": system_name,
                "chain_index": str(chain_idx),
                "repeat_index": str(repeat_idx),
                "associated_peaa_residue": f"repeat_{repeat_idx:03d}_PEAA",
                "residue_name": RESNAME_MAP["NA"],
            })
            for atom in sodium.children:
                atom.role = PrimitiveRole.PARTICLE
                atom.metadata.update(sodium.metadata)

            sodium_segment = Primitive(
                label=f"sodium_chain{chain_idx:04d}_repeat{repeat_idx:03d}",
                role=PrimitiveRole.SEGMENT,
            )
            sodium_segment.metadata.update(sodium.metadata)
            sodium_segment.attach_child(sodium)
            sodium_segments.append(sodium_segment)
            sodium_count += 1

    # Attach ions after all polymer chains so MuPT's PDB-compatible global
    # residue numbering assigns polymer repeat units contiguously up to the
    # 9999-residue PDB chain rollover before numbering counterions.
    for sodium_segment in sodium_segments:
        universe.attach_child(sodium_segment)

    return BuiltIonomerSystem(
        primitive=universe,
        system_name=system_name,
        chain_sequences=chain_sequences,
        sodium_count=sodium_count,
        sdf_paths=[],
    )



## 4. Build Systems

For production-size builds, run one system at a time unless you know your workstation has enough memory.

In [5]:
built_systems = []
for system_name in tqdm(SELECTED_SYSTEMS, desc="building selected systems", unit="system"):
    built = build_ionomer_system(system_name, n_chains=N_CHAINS, random_seed=RANDOM_SEED)
    built_systems.append(built)

    sequence = expanded_sequence(system_name)
    print(f"{system_name}: chains={N_CHAINS}")
    print(f"  repeat units per chain: {len(sequence)}")
    print(f"  PEAA per chain: {sequence.count('PEAA')}")
    print(f"  sodium ions: {built.sodium_count}")
    print(f"  total particles: {len(built.primitive.leaves)}")

building selected systems:   0%|          | 0/1 [00:00<?, ?system/s]

building m3 chains:   0%|          | 0/100 [00:00<?, ?chain/s]

Attempting to infer internal connections automatically from given topology; user should verify the connections assigned make sense!
Attempting to infer internal connections automatically from given topology; user should verify the connections assigned make sense!
Attempting to infer internal connections automatically from given topology; user should verify the connections assigned make sense!
Attempting to infer internal connections automatically from given topology; user should verify the connections assigned make sense!
Attempting to infer internal connections automatically from given topology; user should verify the connections assigned make sense!
Attempting to infer internal connections automatically from given topology; user should verify the connections assigned make sense!
Attempting to infer internal connections automatically from given topology; user should verify the connections assigned make sense!
Attempting to infer internal connections automatically from given topology; 

m3: chains=100
  repeat units per chain: 36
  PEAA per chain: 12
  sodium ions: 1200
  total particles: 39800


## 5. Coordinate Diagnostics

The starting coordinates are not a melt-packing algorithm. They are non-overlapping random-walk coordinates intended for OpenMM minimization and vacuum collapse before optional periodic NPT.

In [6]:
def primitive_positions(primitive: Primitive) -> np.ndarray:
    """Collect leaf-particle coordinates from a MuPT primitive."""
    return np.vstack([np.array(leaf.shape.centroid, dtype=float) for leaf in primitive.leaves if leaf.shape is not None])


def minimum_pair_distance(positions: np.ndarray) -> float:
    """Return nearest-neighbor distance without allocating an O(N^2) matrix."""
    from scipy.spatial import cKDTree

    distances, _ = cKDTree(positions).query(positions, k=2)
    return float(np.min(distances[:, 1]))


for built in built_systems:
    positions = primitive_positions(built.primitive)
    span = positions.max(axis=0) - positions.min(axis=0)
    min_distance = minimum_pair_distance(positions)
    print(f"{built.system_name}: minimum atom distance = {min_distance:.3f} A")
    print(f"{built.system_name}: coordinate span = {span[0]:.1f} x {span[1]:.1f} x {span[2]:.1f} A")
    if min_distance < MIN_ALLOWED_DISTANCE_A:
        print(
            f"Warning: {built.system_name} is below the requested initial-distance threshold. "
            "Continuing to OpenMM minimization/collapse; inspect the structure if minimization fails."
        )


m3: minimum atom distance = 0.035 A
m3: coordinate span = 303.2 x 288.9 x 293.8 A


## 6. Export Role-Aware SDF Files

Each polymer chain and sodium ion is still exported as its own SDF molecule record, but the notebook now stores those records in one multi-record SDF file per system instead of one file per segment. Sodium residues remain associated with their PEAA repeat through metadata, not covalent bonds.

A single multi-conformer SDF entry is not enough for the repeated sodium ions in this workflow: OpenFF treats multiple conformers as alternate coordinates for one chemical graph, not as distinct molecule instances with different residue numbers. That means multi-record SDF is the practical file-count reduction here, while true conformer-sharing would still need a separate per-instance coordinate/metadata transport layer.

In [7]:
MUPT_ATOM_PROPS_FOR_SDF = [
    # RDKit SDF reload does not preserve AtomPDBResidueInfo directly, so write
    # both PDB-style atom metadata and MuPT hierarchy metadata as atom-property
    # lists for downstream OpenFF/OpenMM notebooks.
    "chain_id",
    "residue_id",
    "residue_name",
    "mupt_segment_index",
    "mupt_segment_label",
    "mupt_residue_index",
    "mupt_residue_label",
    "mupt_particle_index",
    "mupt_particle_label",
]


def prepare_mupt_sdf_atom_props(mol: Chem.Mol) -> None:
    """Store MuPT atom props as SDF atom-property lists before writing."""
    for prop_name in MUPT_ATOM_PROPS_FOR_SDF:
        Chem.CreateAtomStringPropertyList(mol, prop_name)


def segment_label_from_mol(mol: Chem.Mol, mol_idx: int) -> str:
    """Return the MuPT segment label stored on an exported RDKit molecule."""
    if mol.HasProp("mupt_segment_label"):
        return mol.GetProp("mupt_segment_label")
    if mol.GetNumAtoms() and mol.GetAtomWithIdx(0).HasProp("mupt_segment_label"):
        return mol.GetAtomWithIdx(0).GetProp("mupt_segment_label")
    return f"mol_{mol_idx:05d}"


for idx, built in enumerate(tqdm(built_systems, desc="exporting systems", unit="system")):
    sdf_dir = OUTPUT_ROOT / built.system_name / "sdf"
    sdf_dir.mkdir(parents=True, exist_ok=True)
    for stale_path in sdf_dir.glob("*.sdf"):
        stale_path.unlink()
    resname_map = {
        residue.label: residue.metadata.get("residue_name", "UNK")
        for segment in built.primitive.children
        for residue in segment.children
    }

    # Canonical MuPT export: one RDKit molecule per SEGMENT, with SAAMR hierarchy
    # metadata and PDB-compatible residue metadata supplied by MuPT.
    rdkit_mols = primitive_to_rdkit_mols(
        built.primitive,
        resname_map=resname_map,
        default_atom_position=np.zeros(3),
    )
    # Coordinates were already assigned at the MuPT primitive level from repeat
    # conformers and random-walk placement, including sodium positions. Avoid
    # full-chain ETKDG here so production-size systems remain tractable.

    sdf_path = sdf_dir / f"{built.system_name}.sdf"
    writer = Chem.SDWriter(str(sdf_path))
    for mol_idx, mol in enumerate(tqdm(rdkit_mols, desc=f"writing {built.system_name} SDF", unit="mol", leave=False)):
        prepare_mupt_sdf_atom_props(mol)
        label = segment_label_from_mol(mol, mol_idx)
        if not mol.HasProp("_Name"):
            mol.SetProp("_Name", label)
        writer.write(mol)
    writer.close()
    sdf_paths = [sdf_path]

    built_systems[idx] = BuiltIonomerSystem(
        primitive=built.primitive,
        system_name=built.system_name,
        chain_sequences=built.chain_sequences,
        sodium_count=built.sodium_count,
        sdf_paths=sdf_paths,
    )
    print(
        f"{built.system_name}: wrote {len(rdkit_mols)} molecule record(s) to multi-record SDF {sdf_path}"
    )


exporting systems:   0%|          | 0/1 [00:00<?, ?system/s]

writing m3 SDF:   0%|          | 0/1300 [00:00<?, ?mol/s]

m3: wrote 1300 SDF file(s) to /home/joelaforet/Shirts-Lab-Linux/mupt-examples/examples_system/role_aware_ionomer_outputs/m3/sdf


## 7. Fast MuPT SDF Validation

This reload uses MuPT metadata only. It avoids expensive bond and shape reconstruction, which matters for production-size SDF sets.

In [8]:
for built in built_systems:
    reconstructed = primitive_from_mupt_sdf(
        built.sdf_paths,
        reconstruct_bonds=False,
        reconstruct_shapes=False,
    )
    chain_segments = [segment for segment in reconstructed.children if str(segment.label).startswith("chain_")]
    sodium_segments = [segment for segment in reconstructed.children if str(segment.label).startswith("sodium_")]
    peaa_residues = [
        residue
        for segment in chain_segments
        for residue in segment.children
        if str(residue.label).endswith("_PEAA")
    ]

    assert len(chain_segments) == N_CHAINS
    assert len(sodium_segments) == built.sodium_count
    assert len(peaa_residues) == built.sodium_count
    print(
        f"{built.system_name}: reconstructed chains={len(chain_segments)}, "
        f"PEAA={len(peaa_residues)}, sodium={len(sodium_segments)}"
    )

m3: reconstructed chains=100, PEAA=1200, sodium=1200


## 8. OpenFF/OpenMM Setup

The cells below mirror the companion OpenFF/OpenMM notebook: use OpenFF NAGL GNN charges for polymer chains, avoid AM1-BCC fallback, minimize and briefly collapse in vacuum, then optionally wrap a padded periodic box for NPT. Keep `RUN_OPENFF_PARAMETERIZATION = False` until the smoke-test SDFs look right.

OpenFF can preserve multiple conformers on one `Molecule`, but `Topology.from_molecules(...)` still interprets them as alternate coordinates for the same molecule identity. For this ionomer workflow we therefore read every record from the multi-record SDF and create one OpenFF molecule instance per record.

In [9]:
try:
    from openff.interchange import Interchange
    from openff.toolkit import ForceField, Molecule, Topology
    from openff.toolkit.utils import ToolkitRegistry
    from openff.units import unit as off_unit
    OPENFF_AVAILABLE = True
except ModuleNotFoundError as exc:
    OPENFF_AVAILABLE = False
    OPENFF_IMPORT_ERROR = exc

NAGL_AVAILABLE = False
NAGL_IMPORT_ERROR = None
if OPENFF_AVAILABLE:
    try:
        from openff.toolkit.utils.nagl_wrapper import NAGLToolkitWrapper

        NAGL_AVAILABLE = NAGLToolkitWrapper.is_available()
        if not NAGL_AVAILABLE:
            NAGL_IMPORT_ERROR = RuntimeError("OpenFF NAGL backend is unavailable; install openff-nagl")
    except ModuleNotFoundError as exc:
        NAGL_IMPORT_ERROR = exc

try:
    import openmm
    from openmm import LangevinMiddleIntegrator, MonteCarloBarostat, XmlSerializer
    from openmm import unit as omm_unit
    from openmm.app import DCDReporter, PDBFile, StateDataReporter
    OPENMM_AVAILABLE = True
except ModuleNotFoundError as exc:
    OPENMM_AVAILABLE = False
    OPENMM_IMPORT_ERROR = exc

FORCE_FIELD = "openff-2.2.1.offxml"
PARTIAL_CHARGE_METHOD = "openff-gnn-am1bcc-1.0.0.pt"
RUN_OPENFF_PARAMETERIZATION = False
RUN_VACUUM_COLLAPSE = RUN_OPENFF_PARAMETERIZATION and OPENMM_AVAILABLE
RUN_PERIODIC_NPT = False
DEFAULT_TEMPERATURE_K = 373.0
VACUUM_COLLAPSE_TEMPERATURE_K = 800.0
VACUUM_COLLAPSE_MAX_DURATION_NS = 0.50
VACUUM_COLLAPSE_CHUNK_DURATION_NS = 0.05
VACUUM_COLLAPSE_MIN_CHUNKS = 2
VACUUM_SPAN_RELATIVE_TOLERANCE = 0.03
VACUUM_TIMESTEP_FS = 2.0
VACUUM_N_FRAMES = 50
PERIODIC_INITIAL_DENSITY_G_CM3 = 0.30
PRODUCTION_START_DENSITY_G_CM3 = 0.85
MELT_PREPARATION_PROTOCOL = [
    {"name": "foam collapse", "target_density_g_cm3": 0.35, "temperature_k": 800.0, "pressure_atm": 200.0, "max_duration_ns": 2.0},
    {"name": "dense packing", "target_density_g_cm3": 0.60, "temperature_k": 650.0, "pressure_atm": 100.0, "max_duration_ns": 4.0},
    {"name": "melt approach", "target_density_g_cm3": 0.80, "temperature_k": 500.0, "pressure_atm": 25.0, "max_duration_ns": 6.0},
    {"name": "production-start relaxation", "target_density_g_cm3": PRODUCTION_START_DENSITY_G_CM3, "temperature_k": DEFAULT_TEMPERATURE_K, "pressure_atm": 1.0, "max_duration_ns": 5.0},
]
PERIODIC_NPT_CHUNK_DURATION_NS = 0.02
PERIODIC_TIMESTEP_FS = 2.0
PERIODIC_N_FRAMES = 200
PERIODIC_PADDING_NM = 0.4
MIN_PERIODIC_BOX_CUTOFF_MULTIPLIER = 2.2

print(f"OpenFF available: {OPENFF_AVAILABLE}")
if not OPENFF_AVAILABLE:
    print(f"  {OPENFF_IMPORT_ERROR}")
print(f"OpenFF NAGL available: {NAGL_AVAILABLE}")
if not NAGL_AVAILABLE and NAGL_IMPORT_ERROR is not None:
    print(f"  {NAGL_IMPORT_ERROR}")
print(f"OpenMM available: {OPENMM_AVAILABLE}")
if not OPENMM_AVAILABLE:
    print(f"  {OPENMM_IMPORT_ERROR}")
print(f"Run OpenFF parameterization: {RUN_OPENFF_PARAMETERIZATION}")

OpenFF available: True
OpenFF NAGL available: True
OpenMM available: True
Run OpenFF parameterization: True


In [10]:
def transfer_rdkit_metadata_to_openff(rdkit_mol: Chem.Mol, off_mol) -> None:
    """Copy SDF atom-property metadata into OpenFF atom metadata."""
    for rd_atom, off_atom in zip(rdkit_mol.GetAtoms(), off_mol.atoms):
        props = rd_atom.GetPropsAsDict(includePrivate=True, includeComputed=False)
        off_atom.metadata.update({
            "residue_name": str(props.get("residue_name", "UNK")),
            "residue_number": str(props.get("residue_id", props.get("mupt_residue_index", "1"))),
            "chain_id": str(props.get("chain_id", "A")),
            "atom_name": f"{rd_atom.GetSymbol()}{rd_atom.GetIdx() + 1}",
        })


def load_rdkit_sdf_records(paths: list[Path]) -> list[Chem.Mol]:
    """Load every SDF molecule record without stripping explicit hydrogens."""
    records = []
    for path in paths:
        supplier = Chem.SDMolSupplier(str(path), removeHs=False, sanitize=False)
        for record_idx, mol in enumerate(supplier):
            if mol is None:
                raise ValueError(f"Could not read record {record_idx} from {path}")
            sanitized = Chem.Mol(mol)
            Chem.SanitizeMol(sanitized)
            records.append(sanitized)
    return records


openff_molecules_by_system = {}
if OPENFF_AVAILABLE:
    for built in built_systems:
        off_molecules = []
        for rdkit_mol in load_rdkit_sdf_records(built.sdf_paths):
            off_mol = Molecule.from_rdkit(
                rdkit_mol,
                allow_undefined_stereo=True,
                hydrogens_are_explicit=True,
            )
            transfer_rdkit_metadata_to_openff(rdkit_mol, off_mol)
            off_molecules.append(off_mol)
        openff_molecules_by_system[built.system_name] = off_molecules
        print(f"{built.system_name}: created {len(off_molecules)} OpenFF molecule(s)")
else:
    print("Skipping OpenFF molecule conversion because openff-toolkit is unavailable.")

m3: created 1300 OpenFF molecule(s)


In [11]:
def is_sodium_molecule(off_mol: Molecule) -> bool:
    """Return True for the one-atom sodium counterion molecules."""
    return off_mol.n_atoms == 1 and off_mol.atom(0).symbol == "Na"


interchanges_by_system = {}

if OPENFF_AVAILABLE and NAGL_AVAILABLE and RUN_OPENFF_PARAMETERIZATION:
    ff = ForceField(FORCE_FIELD)
    nagl_registry = ToolkitRegistry([NAGLToolkitWrapper()])

    for built in built_systems:
        off_molecules = openff_molecules_by_system[built.system_name]
        topology = Topology.from_molecules(off_molecules)
        unique_charge_molecules = []
        unique_molecules = list(topology.unique_molecules)
        print(
            f"{built.system_name}: parameterizing {len(unique_molecules)} unique molecule type(s) "
            f"across {len(off_molecules)} total molecules"
        )

        for unique_idx, unique_mol in enumerate(unique_molecules, start=1):
            if is_sodium_molecule(unique_mol):
                print(
                    f"{built.system_name}: assigning sodium charges for unique molecule "
                    f"{unique_idx}/{len(unique_molecules)}"
                )
                unique_mol.partial_charges = [1.0] * off_unit.elementary_charge
            else:
                print(
                    f"{built.system_name}: assigning NAGL charges for unique molecule "
                    f"{unique_idx}/{len(unique_molecules)} ({unique_mol.n_atoms} atoms)"
                )
                unique_mol.assign_partial_charges(
                    partial_charge_method=PARTIAL_CHARGE_METHOD,
                    toolkit_registry=nagl_registry,
                )
            unique_charge_molecules.append(unique_mol)

        interchange = ff.create_interchange(
            topology,
            charge_from_molecules=unique_charge_molecules,
        )
        interchange.box = None
        interchanges_by_system[built.system_name] = interchange
        print(
            f"{built.system_name}: created vacuum interchange with {interchange.topology.n_atoms} atoms "
            f"from {len(unique_charge_molecules)} unique molecule type(s)"
        )
elif OPENFF_AVAILABLE and not NAGL_AVAILABLE:
    print("Parameterization skipped because OpenFF NAGL is unavailable; no AM1-BCC fallback is used.")
else:
    print("Parameterization skipped. Set RUN_OPENFF_PARAMETERIZATION = True after checking the smoke-test SDFs.")

m3: parameterizing 2 unique molecule type(s) across 1300 total molecules
m3: assigning NAGL charges for unique molecule 1/2 (386 atoms)
m3: assigning sodium charges for unique molecule 2/2


/home/joelaforet/miniconda3/envs/mupt-env/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):


m3: created vacuum interchange with 39800 atoms from 2 unique molecule type(s)


## 9. Vacuum Collapse and Optional Periodic NPT

This follows the companion notebook's pragmatic route: collapse in vacuum first to avoid building an enormous sparse PME grid around random-walk starting coordinates. Only enable periodic NPT after inspecting the vacuum result.

In [ ]:
DA_PER_NM3_TO_G_CM3 = 1.0 / 602.214076


def simulation_steps(duration_ns: float, timestep_fs: float) -> int:
    """Convert a duration in ns and timestep in fs to OpenMM steps."""
    if duration_ns < 0:
        raise ValueError("duration_ns must be non-negative")
    if timestep_fs <= 0:
        raise ValueError("timestep_fs must be positive")
    return int(round(duration_ns * 1_000_000.0 / timestep_fs))


def report_interval(total_steps: int, n_frames: int) -> int:
    """Return a reporter interval that saves approximately n_frames frames."""
    if n_frames <= 0:
        raise ValueError("n_frames must be positive")
    return max(1, total_steps // n_frames) if total_steps else 1


def openmm_system_mass_da(system) -> float:
    """Return total OpenMM system mass in daltons."""
    return sum(system.getParticleMass(i).value_in_unit(omm_unit.dalton) for i in range(system.getNumParticles()))


def maximum_nonbonded_cutoff_nm(system) -> float:
    """Return the largest cutoff distance used by OpenMM nonbonded-like forces."""
    cutoffs = []
    for force in system.getForces():
        if hasattr(force, "getCutoffDistance"):
            try:
                cutoffs.append(force.getCutoffDistance().value_in_unit(omm_unit.nanometer))
            except Exception:
                pass
    if not cutoffs:
        raise ValueError("Could not determine a nonbonded cutoff from the OpenMM system")
    return float(max(cutoffs))


def density_from_box_g_cm3(mass_da: float, box_vectors_nm: np.ndarray) -> float:
    """Return mass density from box vectors in nm."""
    volume_nm3 = abs(float(np.linalg.det(box_vectors_nm)))
    return mass_da * DA_PER_NM3_TO_G_CM3 / volume_nm3


def coordinate_span_nm(positions_nm: np.ndarray) -> np.ndarray:
    """Return orthorhombic coordinate span in nm."""
    return positions_nm.max(axis=0) - positions_nm.min(axis=0)


def box_lengths_nm(box_vectors_nm: np.ndarray) -> np.ndarray:
    """Return periodic box vector lengths in nm."""
    return np.array([np.linalg.norm(vector) for vector in box_vectors_nm], dtype=float)


def set_barostat_conditions(simulation, temperature_k: float, pressure_atm: float) -> None:
    """Update integrator and barostat thermodynamic targets in-place."""
    simulation.integrator.setTemperature(temperature_k * omm_unit.kelvin)
    pressure_bar = (pressure_atm * omm_unit.atmosphere).value_in_unit(omm_unit.bar)
    simulation.context.setParameter(MonteCarloBarostat.Temperature(), temperature_k)
    simulation.context.setParameter(MonteCarloBarostat.Pressure(), pressure_bar)


def set_periodic_box_for_initial_density(interchange, mass_da: float, density_g_cm3: float, padding_nm: float):
    """Set the smallest orthorhombic box satisfying density and padding constraints."""
    positions_nm = interchange.positions.m_as(off_unit.nanometer)
    mins = positions_nm.min(axis=0)
    maxs = positions_nm.max(axis=0)
    shifted_positions = positions_nm - mins + padding_nm
    span_lengths = (maxs - mins) + 2 * padding_nm
    target_length = (mass_da * DA_PER_NM3_TO_G_CM3 / density_g_cm3) ** (1.0 / 3.0)
    box_lengths = np.maximum(span_lengths, target_length)
    interchange.positions = shifted_positions * off_unit.nanometer
    interchange.box = np.diag(box_lengths) * off_unit.nanometer
    return box_lengths



def metadata_openmm_topology(off_molecules, box_vectors_nm: np.ndarray | None = None) -> openmm.app.Topology:
    """Build an OpenMM topology grouped by MuPT/PDB atom metadata."""
    topology = openmm.app.Topology()
    chains = {}
    residues = {}
    atom_lookup = {}

    for mol_idx, off_mol in enumerate(off_molecules):
        for atom_idx, off_atom in enumerate(off_mol.atoms):
            chain_id = str(off_atom.metadata.get("chain_id", "A"))
            residue_number = str(off_atom.metadata.get("residue_number", "1"))
            residue_name = str(off_atom.metadata.get("residue_name", "UNK"))
            atom_name = str(off_atom.metadata.get("atom_name", f"{off_atom.symbol}{atom_idx + 1}"))

            chain = chains.get(chain_id)
            if chain is None:
                chain = topology.addChain(chain_id)
                chains[chain_id] = chain

            residue_key = (chain_id, residue_number, residue_name)
            residue = residues.get(residue_key)
            if residue is None:
                residue = topology.addResidue(residue_name, chain, id=residue_number)
                residues[residue_key] = residue

            element = openmm.app.element.get_by_symbol(off_atom.symbol)
            atom_lookup[(mol_idx, atom_idx)] = topology.addAtom(atom_name, element, residue)

        for bond in off_mol.bonds:
            topology.addBond(
                atom_lookup[(mol_idx, bond.atom1_index)],
                atom_lookup[(mol_idx, bond.atom2_index)],
            )

    if box_vectors_nm is not None:
        topology.setPeriodicBoxVectors(box_vectors_nm * omm_unit.nanometer)

    return topology


def write_production_start_files(
    simulation,
    off_molecules,
    output_dir: Path,
    system_name: str,
    system_mass_da: float,
    steps_run: int,
    timestep_fs: float,
    temperature_k: float,
    pressure_atm: float,
    density_g_cm3: float,
) -> dict[str, Path]:
    """Write restart-ready coordinates, box vectors, and metadata for production MD."""
    production_dir = output_dir / "production_start"
    production_dir.mkdir(parents=True, exist_ok=True)

    state = simulation.context.getState(
        getEnergy=True,
        getPositions=True,
        getVelocities=True,
        enforcePeriodicBox=True,
    )
    positions_nm = state.getPositions(asNumpy=True).value_in_unit(omm_unit.nanometer)
    box_vectors_nm = state.getPeriodicBoxVectors(asNumpy=True).value_in_unit(omm_unit.nanometer)
    density = density_from_box_g_cm3(system_mass_da, box_vectors_nm)
    lengths_nm = box_lengths_nm(box_vectors_nm)

    topology = metadata_openmm_topology(off_molecules, box_vectors_nm=box_vectors_nm)
    pdb_path = production_dir / f"{system_name}_production_start.pdb"
    state_path = production_dir / f"{system_name}_production_start_state.xml"
    checkpoint_path = production_dir / f"{system_name}_production_start.chk"
    arrays_path = production_dir / f"{system_name}_production_start_arrays.npz"
    manifest_path = production_dir / f"{system_name}_production_start_manifest.json"

    with pdb_path.open("w") as handle:
        PDBFile.writeFile(topology, state.getPositions(asNumpy=True), handle)
    state_path.write_text(XmlSerializer.serialize(state))
    simulation.saveCheckpoint(str(checkpoint_path))
    np.savez_compressed(
        arrays_path,
        positions_nm=positions_nm,
        box_vectors_nm=box_vectors_nm,
        box_lengths_nm=lengths_nm,
    )

    manifest = {
        "system_name": system_name,
        "intended_use": "starting coordinates for subsequent production MD",
        "steps_run": int(steps_run),
        "elapsed_ns": float(steps_run * timestep_fs / 1_000_000.0),
        "temperature_k": float(temperature_k),
        "pressure_atm": float(pressure_atm),
        "target_density_g_cm3": float(density_g_cm3),
        "final_density_g_cm3": float(density),
        "mass_da": float(system_mass_da),
        "box_lengths_nm": [float(x) for x in lengths_nm],
        "box_vectors_nm": box_vectors_nm.tolist(),
        "files": {
            "pdb": str(pdb_path.relative_to(EXAMPLES_ROOT)),
            "state_xml": str(state_path.relative_to(EXAMPLES_ROOT)),
            "checkpoint": str(checkpoint_path.relative_to(EXAMPLES_ROOT)),
            "arrays_npz": str(arrays_path.relative_to(EXAMPLES_ROOT)),
        },
    }
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")

    return {
        "pdb": pdb_path,
        "state_xml": state_path,
        "checkpoint": checkpoint_path,
        "arrays_npz": arrays_path,
        "manifest": manifest_path,
    }


def attach_openmm_reporters(simulation, trajectory_path: Path, state_data_path: Path, interval: int, include_density: bool) -> None:
    """Attach trajectory and state-data reporters for later analysis."""
    simulation.reporters.append(DCDReporter(str(trajectory_path), interval))
    simulation.reporters.append(
        StateDataReporter(
            str(state_data_path),
            reportInterval=interval,
            step=True,
            time=True,
            potentialEnergy=True,
            kineticEnergy=True,
            temperature=True,
            volume=include_density,
            density=include_density,
            speed=True,
        )
    )


if interchanges_by_system and OPENMM_AVAILABLE and RUN_VACUUM_COLLAPSE:
    vacuum_max_steps = simulation_steps(VACUUM_COLLAPSE_MAX_DURATION_NS, VACUUM_TIMESTEP_FS)
    vacuum_chunk_steps = max(1, simulation_steps(VACUUM_COLLAPSE_CHUNK_DURATION_NS, VACUUM_TIMESTEP_FS))
    vacuum_interval = report_interval(vacuum_max_steps, VACUUM_N_FRAMES)
    periodic_chunk_steps = max(1, simulation_steps(PERIODIC_NPT_CHUNK_DURATION_NS, PERIODIC_TIMESTEP_FS))
    periodic_max_steps = sum(
        simulation_steps(stage["max_duration_ns"], PERIODIC_TIMESTEP_FS)
        for stage in MELT_PREPARATION_PROTOCOL
    )
    periodic_interval = report_interval(periodic_max_steps, PERIODIC_N_FRAMES)

    for built in built_systems:
        interchange = interchanges_by_system[built.system_name]
        openmm_dir = OUTPUT_ROOT / built.system_name / "OpenMM"
        openmm_dir.mkdir(parents=True, exist_ok=True)

        first_stage = MELT_PREPARATION_PROTOCOL[0]
        pressure = first_stage["pressure_atm"] * omm_unit.atmosphere
        vacuum_temperature = VACUUM_COLLAPSE_TEMPERATURE_K * omm_unit.kelvin
        periodic_temperature = first_stage["temperature_k"] * omm_unit.kelvin
        vacuum_time_step = VACUUM_TIMESTEP_FS * omm_unit.femtosecond
        periodic_time_step = PERIODIC_TIMESTEP_FS * omm_unit.femtosecond
        friction = 10.0 / omm_unit.picosecond
        system_name = f"ionomer_{built.system_name}"

        print(f"{built.system_name}: creating non-periodic vacuum OpenMM simulation")
        vacuum_integrator = LangevinMiddleIntegrator(vacuum_temperature, friction, vacuum_time_step)
        vacuum_simulation = interchange.to_openmm_simulation(
            integrator=vacuum_integrator,
            combine_nonbonded_forces=False,
        )
        system_mass_da = openmm_system_mass_da(vacuum_simulation.system)
        print(f"{built.system_name}: running vacuum minimization")
        vacuum_simulation.minimizeEnergy()
        if vacuum_max_steps > 0:
            vacuum_dcd_path = openmm_dir / f"{system_name}_vacuum_trajectory.dcd"
            vacuum_state_data_path = openmm_dir / f"{system_name}_vacuum_state_data.csv"
            attach_openmm_reporters(vacuum_simulation, vacuum_dcd_path, vacuum_state_data_path, vacuum_interval, include_density=False)
            print(
                f"{built.system_name}: running vacuum collapse until coordinate span stabilizes "
                f"or {VACUUM_COLLAPSE_MAX_DURATION_NS} ns at {VACUUM_COLLAPSE_TEMPERATURE_K} K"
            )
            vacuum_steps_run = 0
            previous_span_volume = None
            vacuum_chunk_idx = 0
            while vacuum_steps_run < vacuum_max_steps:
                steps_this_chunk = min(vacuum_chunk_steps, vacuum_max_steps - vacuum_steps_run)
                vacuum_simulation.step(steps_this_chunk)
                vacuum_steps_run += steps_this_chunk
                vacuum_chunk_idx += 1
                state = vacuum_simulation.context.getState(getPositions=True)
                positions_nm = state.getPositions(asNumpy=True).value_in_unit(omm_unit.nanometer)
                span = coordinate_span_nm(positions_nm)
                span_volume = float(np.prod(span))
                elapsed_ns = vacuum_steps_run * VACUUM_TIMESTEP_FS / 1_000_000.0
                relative_change = np.inf if previous_span_volume is None else abs(span_volume - previous_span_volume) / previous_span_volume
                print(
                    f"{built.system_name}: vacuum {elapsed_ns:.4f} ns, "
                    f"span={span[0]:.2f} x {span[1]:.2f} x {span[2]:.2f} nm, "
                    f"span-volume change={relative_change:.3f}"
                )
                if vacuum_chunk_idx >= VACUUM_COLLAPSE_MIN_CHUNKS and relative_change < VACUUM_SPAN_RELATIVE_TOLERANCE:
                    print(f"{built.system_name}: vacuum collapse converged by span-volume criterion")
                    break
                previous_span_volume = span_volume
            print(f"{built.system_name}: vacuum trajectory saved to {vacuum_dcd_path.relative_to(EXAMPLES_ROOT)}")
            print(f"{built.system_name}: vacuum state data saved to {vacuum_state_data_path.relative_to(EXAMPLES_ROOT)}")

        vacuum_state = vacuum_simulation.context.getState(getPositions=True, getEnergy=True)
        collapsed_positions_nm = vacuum_state.getPositions(asNumpy=True).value_in_unit(omm_unit.nanometer)
        interchange.positions = collapsed_positions_nm * off_unit.nanometer

        vacuum_topology_path = openmm_dir / f"{system_name}_vacuum_topology.pdb"
        vacuum_system_path = openmm_dir / f"{system_name}_vacuum_system.xml"
        vacuum_integrator_path = openmm_dir / f"{system_name}_vacuum_integrator.xml"
        vacuum_state_path = openmm_dir / f"{system_name}_vacuum_state.xml"
        with vacuum_topology_path.open("w") as handle:
            PDBFile.writeFile(metadata_openmm_topology(openff_molecules_by_system[built.system_name]), vacuum_state.getPositions(asNumpy=True), handle)
        vacuum_system_path.write_text(XmlSerializer.serialize(vacuum_simulation.system))
        vacuum_integrator_path.write_text(XmlSerializer.serialize(vacuum_integrator))
        vacuum_state_path.write_text(XmlSerializer.serialize(vacuum_state))

        print(f"{built.system_name}: vacuum potential energy: {vacuum_state.getPotentialEnergy()}")
        print(f"{built.system_name}: serialized vacuum OpenMM components:")
        for output_path in (vacuum_topology_path, vacuum_system_path, vacuum_integrator_path, vacuum_state_path):
            print(f"  {output_path.relative_to(EXAMPLES_ROOT)}")

        if RUN_PERIODIC_NPT:
            box_lengths = set_periodic_box_for_initial_density(
                interchange,
                mass_da=system_mass_da,
                density_g_cm3=PERIODIC_INITIAL_DENSITY_G_CM3,
                padding_nm=PERIODIC_PADDING_NM,
            )
            print(
                f"{built.system_name}: starting periodic NPT from box lengths "
                f"{box_lengths[0]:.2f} x {box_lengths[1]:.2f} x {box_lengths[2]:.2f} nm "
                f"toward production-start density {PRODUCTION_START_DENSITY_G_CM3:.2f} g/cm^3"
            )
            periodic_integrator = LangevinMiddleIntegrator(periodic_temperature, friction, periodic_time_step)
            periodic_simulation = interchange.to_openmm_simulation(
                integrator=periodic_integrator,
                combine_nonbonded_forces=False,
                additional_forces=[MonteCarloBarostat(pressure, periodic_temperature, 25)],
            )
            nonbonded_cutoff_nm = maximum_nonbonded_cutoff_nm(periodic_simulation.system)
            minimum_box_length_nm = MIN_PERIODIC_BOX_CUTOFF_MULTIPLIER * nonbonded_cutoff_nm
            print(
                f"{built.system_name}: cutoff guard stops NPT before the minimum box length "
                f"falls below {minimum_box_length_nm:.2f} nm "
                f"({MIN_PERIODIC_BOX_CUTOFF_MULTIPLIER:.1f} x {nonbonded_cutoff_nm:.2f} nm cutoff)"
            )
            print(f"{built.system_name}: running periodic minimization before NPT dynamics")
            periodic_simulation.minimizeEnergy()
            periodic_dcd_path = openmm_dir / f"{system_name}_periodic_npt_trajectory.dcd"
            periodic_state_data_path = openmm_dir / f"{system_name}_periodic_npt_state_data.csv"
            attach_openmm_reporters(periodic_simulation, periodic_dcd_path, periodic_state_data_path, periodic_interval, include_density=True)

            steps_run = 0
            current_density = 0.0
            stopped_by_cutoff_guard = False
            final_stage = MELT_PREPARATION_PROTOCOL[-1]
            final_temperature_k = float(final_stage["temperature_k"])
            final_pressure_atm = float(final_stage["pressure_atm"])
            for stage_idx, stage in enumerate(MELT_PREPARATION_PROTOCOL, start=1):
                stage_name = str(stage["name"])
                target_density = float(stage["target_density_g_cm3"])
                stage_temperature_k = float(stage["temperature_k"])
                stage_pressure_atm = float(stage["pressure_atm"])
                stage_max_steps = simulation_steps(float(stage["max_duration_ns"]), PERIODIC_TIMESTEP_FS)
                stage_steps_run = 0
                set_barostat_conditions(periodic_simulation, stage_temperature_k, stage_pressure_atm)
                print(
                    f"{built.system_name}: stage {stage_idx}/{len(MELT_PREPARATION_PROTOCOL)} '{stage_name}' "
                    f"until density {target_density:.2f} g/cm^3 "
                    f"at {stage_temperature_k:.0f} K, {stage_pressure_atm:.1f} atm"
                )
                while stage_steps_run < stage_max_steps and current_density < target_density:
                    steps_this_chunk = min(periodic_chunk_steps, stage_max_steps - stage_steps_run)
                    periodic_simulation.step(steps_this_chunk)
                    steps_run += steps_this_chunk
                    stage_steps_run += steps_this_chunk
                    state = periodic_simulation.context.getState(getPositions=True)
                    box_vectors_nm = state.getPeriodicBoxVectors(asNumpy=True).value_in_unit(omm_unit.nanometer)
                    lengths = box_lengths_nm(box_vectors_nm)
                    if np.min(lengths) <= minimum_box_length_nm:
                        stopped_by_cutoff_guard = True
                        print(
                            f"{built.system_name}: stopping NPT because the minimum box length "
                            f"{np.min(lengths):.2f} nm reached the cutoff guard {minimum_box_length_nm:.2f} nm"
                        )
                        break
                    current_density = density_from_box_g_cm3(system_mass_da, box_vectors_nm)
                    elapsed_ns = steps_run * PERIODIC_TIMESTEP_FS / 1_000_000.0
                    print(
                        f"{built.system_name}: NPT {elapsed_ns:.4f} ns, stage='{stage_name}', "
                        f"density={current_density:.3f} g/cm^3, "
                        f"box={lengths[0]:.2f} x {lengths[1]:.2f} x {lengths[2]:.2f} nm"
                    )
                if stopped_by_cutoff_guard:
                    break
                if current_density >= target_density:
                    print(f"{built.system_name}: reached {stage_name} density threshold ({current_density:.3f} g/cm^3)")
                else:
                    print(
                        f"{built.system_name}: stage '{stage_name}' hit its safety cap at "
                        f"{current_density:.3f} g/cm^3; moving to the next protocol stage"
                    )
                if current_density >= PRODUCTION_START_DENSITY_G_CM3:
                    print(f"{built.system_name}: production-start density threshold reached")
                    break

            print(f"{built.system_name}: periodic trajectory saved to {periodic_dcd_path.relative_to(EXAMPLES_ROOT)}")
            print(f"{built.system_name}: periodic state data saved to {periodic_state_data_path.relative_to(EXAMPLES_ROOT)}")
            if current_density < PRODUCTION_START_DENSITY_G_CM3:
                print(
                    f"{built.system_name}: warning: final density {current_density:.3f} g/cm^3 "
                    f"is below production-start target {PRODUCTION_START_DENSITY_G_CM3:.3f} g/cm^3"
                )
            periodic_state = periodic_simulation.context.getState(getEnergy=True, getPositions=True, getVelocities=True)
            periodic_state_path = openmm_dir / f"{system_name}_periodic_npt_state.xml"
            periodic_state_path.write_text(XmlSerializer.serialize(periodic_state))
            production_files = write_production_start_files(
                periodic_simulation,
                openff_molecules_by_system[built.system_name],
                openmm_dir,
                system_name,
                system_mass_da,
                steps_run,
                PERIODIC_TIMESTEP_FS,
                final_temperature_k,
                final_pressure_atm,
                PRODUCTION_START_DENSITY_G_CM3,
            )
            print(f"{built.system_name}: periodic NPT potential energy: {periodic_state.getPotentialEnergy()}")
            print(f"{built.system_name}: production-start files:")
            for output_path in production_files.values():
                print(f"  {output_path.relative_to(EXAMPLES_ROOT)}")
else:
    print("OpenMM run skipped because no Interchange was created or RUN_VACUUM_COLLAPSE is False.")


m3: creating non-periodic vacuum OpenMM simulation
m3: running vacuum minimization
m3: running vacuum collapse until coordinate span stabilizes or 0.5 ns at 800.0 K
m3: vacuum 0.0500 ns, span=27.74 x 25.31 x 26.18 nm, span-volume change=inf
m3: vacuum 0.1000 ns, span=26.78 x 25.46 x 24.52 nm, span-volume change=0.091
m3: vacuum 0.1500 ns, span=26.37 x 24.51 x 24.76 nm, span-volume change=0.043
m3: vacuum 0.2000 ns, span=25.19 x 23.85 x 24.62 nm, span-volume change=0.076
m3: vacuum 0.2500 ns, span=23.45 x 23.43 x 23.90 nm, span-volume change=0.112
m3: vacuum 0.3000 ns, span=23.45 x 22.95 x 23.98 nm, span-volume change=0.017
m3: vacuum collapse converged by span-volume criterion
m3: vacuum trajectory saved to examples_system/role_aware_ionomer_outputs/m3/OpenMM/ionomer_m3_vacuum_trajectory.dcd
m3: vacuum state data saved to examples_system/role_aware_ionomer_outputs/m3/OpenMM/ionomer_m3_vacuum_state_data.csv
m3: vacuum potential energy: -457460.91015625 kJ/mol
m3: serialized vacuum OpenM

## 10. Notes for Production Runs

For the production target, set `USE_PRODUCTION_SIZE = True` and run one system at a time. The expected sodium counts are 9,600 for `m3`, 5,600 for `m5`, and 4,000 for `m7`. The vacuum/NPT protocol is intended to generate dense, physically relaxed starting coordinates for later production MD rather than to be the final scientific trajectory.

The melt-preparation protocol switches automatically by density: aggressive high-temperature/high-pressure NPT is used only while the system is very dilute, then the protocol relaxes toward the target production-start density. Duration settings are safety caps; density thresholds determine normal stage transitions.

Initial chain placement also self-tunes: if the generated coordinates fail the nearest-neighbor distance check, the build cell increases the chain-start grid spacing and rebuilds the current system automatically.

When `RUN_PERIODIC_NPT = True`, the notebook writes a `production_start/` directory containing a boxed PDB, OpenMM `State` XML, binary checkpoint, compressed NumPy arrays for positions and box vectors, and a JSON manifest with the final density. Use those files to start subsequent simulations at the desired temperature, pressure, and ensemble.
